# Agente de finanzas personales

Proyecto final de SI3003 Inteligencia Artificial. El agente consultará datos financieros sintéticos mediante tres herramientas de solo lectura implementadas con pandas.

## 1. Objetivo y alcance

El prototipo responderá consultas mensuales sobre ingresos, gastos y alertas financieras. No modificará el archivo de datos, no accederá a cuentas bancarias y tendrá un máximo de tres llamadas a herramientas por consulta.

## 2. Configuración del entorno

En esta sección se importarán las dependencias y se configurará de forma privada el proveedor del modelo.

In [1]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "finanzas_personales_sinteticas.xlsx"
MAX_TOOL_CALLS = 3

print(f"Directorio del proyecto: {PROJECT_DIR}")
print(f"Archivo de datos configurado: {DATA_PATH}")

Directorio del proyecto: /Users/laurafrancoberrio/Documents/Laura Eafit /Semestre 8/Inteligencia Artificial /Proyecto Final IA
Archivo de datos configurado: /Users/laurafrancoberrio/Documents/Laura Eafit /Semestre 8/Inteligencia Artificial /Proyecto Final IA/data/finanzas_personales_sinteticas.xlsx


## 3. Carga y validación de datos

Pendiente: cargar las hojas de ingresos y gastos con pandas y validar columnas, fechas, montos, valores faltantes y duplicados. La fuente original se conservará sin modificaciones.

In [2]:
ingresos = pd.read_excel(
    DATA_PATH,
    sheet_name="Ingresos",
    parse_dates=["Fecha"]
)

gastos = pd.read_excel(
    DATA_PATH,
    sheet_name="Gastos",
    parse_dates=["Fecha"]
)

La base de datos utilizada fue generada artificialmente, por lo que no es necesaria una validación de los datos

## 4. Herramientas financieras

Se implementarán y probarán individualmente las siguientes herramientas:

1. `resumen_mensual(mes)`
2. `analizar_gastos(mes, categoria=None)`
3. `detectar_alertas(mes)`

In [3]:
import json
import re
import unicodedata

from langchain_core.tools import tool


def _normalizar_texto(valor: str) -> str:
    """Normaliza texto para comparar sin depender de tildes o mayúsculas."""
    texto = unicodedata.normalize("NFD", str(valor).strip().casefold())
    return "".join(caracter for caracter in texto if unicodedata.category(caracter) != "Mn")


def _a_json(datos: dict) -> str:
    """Convierte una respuesta de herramienta en JSON legible y serializable."""
    return json.dumps(datos, ensure_ascii=False, indent=2)


def _validar_mes(mes: str) -> dict | None:
    """Retorna un error estructurado si el mes no tiene formato YYYY-MM o no existe."""
    if not isinstance(mes, str) or not re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", mes):
        return {
            "ok": False,
            "error": "El mes debe tener el formato YYYY-MM, por ejemplo 2026-03.",
        }

    meses_disponibles = set(ingresos["Mes"].astype(str)) | set(gastos["Mes"].astype(str))
    if mes not in meses_disponibles:
        return {
            "ok": False,
            "error": f"No existen movimientos para el mes {mes}.",
            "meses_disponibles": sorted(meses_disponibles),
        }

    return None


def _resolver_categoria(categoria: str) -> str | None:
    """Encuentra el nombre real de una categoría ignorando tildes y mayúsculas."""
    buscada = _normalizar_texto(categoria)
    for nombre in gastos["Categoría"].dropna().astype(str).unique():
        if _normalizar_texto(nombre) == buscada:
            return nombre
    return None


def _transacciones_serializables(df: pd.DataFrame, limite: int = 5) -> list[dict]:
    """Prepara las transacciones principales para incluirlas en JSON."""
    columnas = ["Fecha", "Categoría", "Subcategoría", "Descripción", "Monto (COP)"]
    resultado = df.nlargest(limite, "Monto (COP)")[columnas].copy()
    resultado["Fecha"] = resultado["Fecha"].dt.strftime("%Y-%m-%d")
    resultado["Monto (COP)"] = resultado["Monto (COP)"].astype(int)
    return resultado.to_dict(orient="records")


In [4]:
@tool
def resumen_mensual(mes: str) -> str:
    """Calcula el resumen financiero de un mes disponible.

    Distingue los egresos de consumo de los aportes registrados como ahorro
    o inversión. No modifica los DataFrames ni el archivo de origen.

    Args:
        mes: Mes que se desea consultar en formato YYYY-MM, por ejemplo 2026-03.
    """
    error = _validar_mes(mes)
    if error:
        return _a_json(error)

    ingresos_mes = ingresos.loc[ingresos["Mes"].astype(str) == mes]
    gastos_mes = gastos.loc[gastos["Mes"].astype(str) == mes]

    total_ingresos = int(ingresos_mes["Monto (COP)"].sum())
    total_egresos = int(gastos_mes["Monto (COP)"].sum())
    es_ahorro = gastos_mes["Categoría"].map(_normalizar_texto) == "ahorro e inversion"
    ahorro_inversion = int(gastos_mes.loc[es_ahorro, "Monto (COP)"].sum())
    gastos_consumo = total_egresos - ahorro_inversion
    balance_caja = total_ingresos - total_egresos
    ahorro_economico = balance_caja + ahorro_inversion

    return _a_json({
        "ok": True,
        "mes": mes,
        "moneda": "COP",
        "total_ingresos": total_ingresos,
        "total_egresos_registrados": total_egresos,
        "gastos_de_consumo": gastos_consumo,
        "ahorro_e_inversion": ahorro_inversion,
        "balance_de_caja": balance_caja,
        "ahorro_economico_estimado": ahorro_economico,
        "porcentaje_egresos_sobre_ingresos": (
            round(total_egresos / total_ingresos * 100, 2) if total_ingresos else None
        ),
        "tasa_de_ahorro_estimada": (
            round(ahorro_economico / total_ingresos * 100, 2) if total_ingresos else None
        ),
        "cantidad_ingresos": int(len(ingresos_mes)),
        "cantidad_egresos": int(len(gastos_mes)),
    })


@tool
def analizar_gastos(mes: str, categoria: str | None = None) -> str:
    """Analiza los gastos de un mes completo o de una categoría específica.

    Entrega totales, porcentajes, frecuencia, promedio, mediana, moda y
    transacciones principales. No modifica los datos.

    Args:
        mes: Mes que se desea consultar en formato YYYY-MM.
        categoria: Categoría opcional, por ejemplo Ocio o Gastos básicos.
    """
    error = _validar_mes(mes)
    if error:
        return _a_json(error)

    gastos_mes = gastos.loc[gastos["Mes"].astype(str) == mes].copy()
    total_mes = int(gastos_mes["Monto (COP)"].sum())

    if categoria is None:
        resumen = (
            gastos_mes.groupby("Categoría")["Monto (COP)"]
            .agg(total="sum", transacciones="size", promedio="mean", mediana="median")
            .sort_values("total", ascending=False)
            .reset_index()
        )
        resumen["porcentaje_del_mes"] = (resumen["total"] / total_mes * 100).round(2)
        resumen["total"] = resumen["total"].astype(int)
        resumen["transacciones"] = resumen["transacciones"].astype(int)
        resumen["promedio"] = resumen["promedio"].round(2)
        resumen["mediana"] = resumen["mediana"].round(2)
        modas = [int(valor) for valor in gastos_mes["Monto (COP)"].mode().head(5)]

        return _a_json({
            "ok": True,
            "mes": mes,
            "moneda": "COP",
            "total_gastos": total_mes,
            "cantidad_transacciones": int(len(gastos_mes)),
            "montos_mas_frecuentes": modas,
            "categorias": resumen.to_dict(orient="records"),
            "transacciones_mas_altas": _transacciones_serializables(gastos_mes),
        })

    categoria_real = _resolver_categoria(categoria)
    if categoria_real is None:
        return _a_json({
            "ok": False,
            "error": f"La categoría '{categoria}' no existe.",
            "categorias_disponibles": sorted(gastos["Categoría"].unique().tolist()),
        })

    seleccion = gastos_mes.loc[gastos_mes["Categoría"] == categoria_real].copy()
    if seleccion.empty:
        return _a_json({
            "ok": False,
            "error": f"No hay gastos de la categoría '{categoria_real}' en {mes}.",
        })

    total_categoria = int(seleccion["Monto (COP)"].sum())
    subcategorias = (
        seleccion.groupby("Subcategoría")["Monto (COP)"]
        .agg(total="sum", transacciones="size")
        .sort_values("total", ascending=False)
        .reset_index()
    )
    subcategorias["total"] = subcategorias["total"].astype(int)
    subcategorias["transacciones"] = subcategorias["transacciones"].astype(int)

    return _a_json({
        "ok": True,
        "mes": mes,
        "categoria": categoria_real,
        "moneda": "COP",
        "total_categoria": total_categoria,
        "porcentaje_del_gasto_mensual": round(total_categoria / total_mes * 100, 2),
        "cantidad_transacciones": int(len(seleccion)),
        "promedio": round(float(seleccion["Monto (COP)"].mean()), 2),
        "mediana": round(float(seleccion["Monto (COP)"].median()), 2),
        "montos_mas_frecuentes": [
            int(valor) for valor in seleccion["Monto (COP)"].mode().head(5)
        ],
        "subcategorias": subcategorias.to_dict(orient="records"),
        "transacciones_mas_altas": _transacciones_serializables(seleccion),
    })


In [5]:
@tool
def detectar_alertas(mes: str) -> str:
    """Detecta alertas financieras mensuales mediante reglas transparentes.

    Revisa déficit de caja, tasa de ahorro estimada inferior al 10 %,
    categorías con aumentos atípicos y transacciones extremas según el
    historial disponible. No modifica los datos.

    Args:
        mes: Mes que se desea revisar en formato YYYY-MM.
    """
    error = _validar_mes(mes)
    if error:
        return _a_json(error)

    ingresos_mes = ingresos.loc[ingresos["Mes"].astype(str) == mes]
    gastos_mes = gastos.loc[gastos["Mes"].astype(str) == mes].copy()
    total_ingresos = int(ingresos_mes["Monto (COP)"].sum())
    total_egresos = int(gastos_mes["Monto (COP)"].sum())
    es_ahorro = gastos_mes["Categoría"].map(_normalizar_texto) == "ahorro e inversion"
    ahorro_inversion = int(gastos_mes.loc[es_ahorro, "Monto (COP)"].sum())
    balance_caja = total_ingresos - total_egresos
    ahorro_economico = balance_caja + ahorro_inversion
    tasa_ahorro = ahorro_economico / total_ingresos * 100 if total_ingresos else None
    alertas = []

    if total_ingresos == 0 and total_egresos > 0:
        alertas.append({
            "nivel": "alta",
            "tipo": "sin_ingresos",
            "detalle": "Hay egresos registrados, pero no hay ingresos en el mes.",
        })
    elif total_egresos > total_ingresos:
        alertas.append({
            "nivel": "alta",
            "tipo": "deficit_de_caja",
            "detalle": f"Los egresos superan los ingresos por {abs(balance_caja):,} COP.",
        })

    if tasa_ahorro is not None and tasa_ahorro < 10:
        alertas.append({
            "nivel": "media",
            "tipo": "ahorro_bajo",
            "detalle": f"La tasa de ahorro estimada es {tasa_ahorro:.2f} %, inferior al umbral de 10 %.",
        })

    totales_mensuales = (
        gastos.groupby(["Mes", "Categoría"])["Monto (COP)"].sum().reset_index()
    )
    actuales = totales_mensuales.loc[totales_mensuales["Mes"].astype(str) == mes]
    historicos = totales_mensuales.loc[totales_mensuales["Mes"].astype(str) != mes]

    for fila in actuales.itertuples(index=False):
        categoria_actual = getattr(fila, "Categoría")
        monto_actual = int(getattr(fila, "_2"))
        serie = historicos.loc[historicos["Categoría"] == categoria_actual, "Monto (COP)"]
        if len(serie) >= 3 and serie.std(ddof=0) > 0:
            limite = serie.mean() + 2 * serie.std(ddof=0)
            if monto_actual > limite:
                alertas.append({
                    "nivel": "media",
                    "tipo": "categoria_atipica",
                    "categoria": categoria_actual,
                    "monto": monto_actual,
                    "detalle": "El gasto de la categoría supera el promedio histórico más dos desviaciones estándar.",
                })

  

    return _a_json({
        "ok": True,
        "mes": mes,
        "reglas": {
            "tasa_minima_ahorro_porcentaje": 10,
            "categoria_atipica": "promedio histórico + 2 desviaciones estándar",
        },
        "cantidad_alertas": len(alertas),
        "alertas": alertas,
        "mensaje": "No se detectaron alertas con las reglas actuales." if not alertas else None,
    })


TOOLS = [resumen_mensual, analizar_gastos, detectar_alertas]
print(f"{len(TOOLS)} herramientas disponibles: {[herramienta.name for herramienta in TOOLS]}")

3 herramientas disponibles: ['resumen_mensual', 'analizar_gastos', 'detectar_alertas']


### Pruebas directas de las herramientas

Estas pruebas se ejecutan antes de conectar el modelo. Así se comprueba que los cálculos funcionan independientemente del agente.

In [6]:
print(resumen_mensual.invoke({"mes": "2026-03"}))
print(analizar_gastos.invoke({"mes": "2026-03", "categoria": "Ocio"}))
print(detectar_alertas.invoke({"mes": "2026-03"}))

{
  "ok": true,
  "mes": "2026-03",
  "moneda": "COP",
  "total_ingresos": 10846570,
  "total_egresos_registrados": 7433450,
  "gastos_de_consumo": 6933450,
  "ahorro_e_inversion": 500000,
  "balance_de_caja": 3413120,
  "ahorro_economico_estimado": 3913120,
  "porcentaje_egresos_sobre_ingresos": 68.53,
  "tasa_de_ahorro_estimada": 36.08,
  "cantidad_ingresos": 9,
  "cantidad_egresos": 69
}
{
  "ok": true,
  "mes": "2026-03",
  "categoria": "Ocio",
  "moneda": "COP",
  "total_categoria": 1271200,
  "porcentaje_del_gasto_mensual": 17.1,
  "cantidad_transacciones": 8,
  "promedio": 158900.0,
  "mediana": 136200.0,
  "montos_mas_frecuentes": [
    40300,
    67600,
    85200,
    92000,
    180400
  ],
  "subcategorias": [
    {
      "Subcategoría": "Salidas con amigos",
      "total": 699600,
      "transacciones": 4
    },
    {
      "Subcategoría": "Cenas familiares",
      "total": 275100,
      "transacciones": 1
    },
    {
      "Subcategoría": "Citas / pareja",
      "total": 1

## 5. Configuración del modelo

La conexión utiliza el endpoint compatible con OpenAI de NVIDIA NIM. La clave y el nombre del modelo se leen desde `.env`; ninguna credencial se escribe en el notebook.

In [31]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

load_dotenv(PROJECT_DIR / ".env", override=True)

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_MODEL = os.getenv("NVIDIA_MODEL", "nvidia/nemotron-3.5-super-vl-120b-a12b")
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

if not NVIDIA_API_KEY:
    raise RuntimeError(
        "Falta NVIDIA_API_KEY. Copia .env.example como .env y agrega tu clave de NVIDIA."
    )

llm = ChatOpenAI(
    model=NVIDIA_MODEL,
    api_key=NVIDIA_API_KEY,
    base_url=NVIDIA_BASE_URL,
    temperature=0,
    max_tokens=2048,
    timeout=60,
    max_retries=2,
)

llm_with_tools = llm.bind_tools(TOOLS)

print(f"Proveedor: NVIDIA NIM | Modelo: {NVIDIA_MODEL}")
print(f"Herramientas conectadas: {[herramienta.name for herramienta in TOOLS]}")

Proveedor: NVIDIA NIM | Modelo: nvidia/nemotron-3.5-super-vl-120b-a12b
Herramientas conectadas: ['resumen_mensual', 'analizar_gastos', 'detectar_alertas']


In [32]:
SYSTEM_PROMPT = """Eres un asistente de análisis de finanzas personales.
Responde siempre en español y utiliza únicamente los datos devueltos por las herramientas.

Herramientas disponibles:
- resumen_mensual: úsala para ingresos, egresos, balance y tasa de ahorro de un mes.
- analizar_gastos: úsala para porcentajes, categorías, frecuencia y transacciones.
- detectar_alertas: úsala para déficit, ahorro bajo y movimientos atípicos.

Reglas:
1. No inventes montos, categorías ni meses.
2. No hagas cálculos financieros de memoria si una herramienta puede hacerlos.
3. Si falta el mes, solicita que el usuario lo indique en formato YYYY-MM.
4. Si una herramienta informa que no hay datos, comunícalo claramente.
5. Distingue gastos de consumo, ahorro o inversión y balance de caja.
6. No afirmes que una alerta constituye asesoría financiera profesional.
7. No solicites ni expongas credenciales o información bancaria.

Después de recibir los resultados de las herramientas, responde de forma concisa.
Prioriza los indicadores principales, las categorías más importantes y las alertas.
No repitas todos los datos devueltos por las herramientas.
"""

print("Prompt del sistema configurado.")

Prompt del sistema configurado.


### Pruebas de conexión y selección de herramientas

La primera prueba confirma que el endpoint responde. La segunda comprueba que el modelo puede elegir una herramienta, pero todavía no la ejecuta; esa ejecución se conectará en el grafo de la siguiente sección.

In [33]:
respuesta_conexion = llm.invoke([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="Responde únicamente con: conexión correcta"),
])
print(respuesta_conexion.content)

conexión correcta


In [34]:
prueba_herramienta = llm_with_tools.invoke([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="¿Cuánto ingresé y gasté en marzo de 2026?"),
])

if prueba_herramienta.tool_calls:
    for llamada in prueba_herramienta.tool_calls:
        print(f"Herramienta elegida: {llamada['name']}")
        print(f"Argumentos: {llamada['args']}")
else:
    print("El modelo respondió sin solicitar una herramienta:")
    print(prueba_herramienta.content)

Herramienta elegida: resumen_mensual
Argumentos: {'mes': '2026-03'}


## 6. Agente con LangGraph

El grafo alterna entre el modelo y las herramientas hasta obtener una respuesta final. El estado cuenta las llamadas ejecutadas y bloquea cualquier solicitud que supere `MAX_TOOL_CALLS`.

In [35]:
from typing import Literal

from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode


class AgentState(MessagesState):
    """Estado del agente con historial y contador de herramientas."""

    tool_calls_used: int


base_tool_node = ToolNode(TOOLS)


def agent_node(state: AgentState) -> dict:
    """Invoca el modelo con el historial actual y el prompt del sistema."""
    messages = [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def tools_node(state: AgentState) -> dict:
    """Ejecuta las herramientas solicitadas y actualiza el contador."""
    last_message = state["messages"][-1]
    requested_calls = len(last_message.tool_calls)
    result = base_tool_node.invoke(state)
    return {
        "messages": result["messages"],
        "tool_calls_used": state.get("tool_calls_used", 0) + requested_calls,
    }


def tool_limit_node(state: AgentState) -> dict:
    """Finaliza de forma controlada cuando se supera el presupuesto."""
    used = state.get("tool_calls_used", 0)
    return {
        "messages": [
            AIMessage(
                content=(
                    f"Se alcanzó el límite de {MAX_TOOL_CALLS} llamadas a herramientas. "
                    f"Se ejecutaron {used}; formula una consulta más específica para continuar."
                )
            )
        ]
    }


def route_after_agent(state: AgentState) -> Literal["tools", "limit", "end"]:
    """Decide si ejecutar herramientas, terminar o aplicar el límite."""
    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None) or []

    if not tool_calls:
        return "end"

    used = state.get("tool_calls_used", 0)
    if used + len(tool_calls) > MAX_TOOL_CALLS:
        return "limit"

    return "tools"


builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_node("limit", tool_limit_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "limit": "limit", "end": END},
)
builder.add_edge("tools", "agent")
builder.add_edge("limit", END)

graph = builder.compile()
print(f"Grafo compilado. Límite: {MAX_TOOL_CALLS} llamadas a herramientas.")

Grafo compilado. Límite: 3 llamadas a herramientas.


## 7. Ejecución y trazabilidad

La función principal mostrará la herramienta seleccionada, sus argumentos, el resultado devuelto y la respuesta final.

In [36]:
def run_agent(query: str, verbose: bool = True) -> dict:
    """Ejecuta el grafo y retorna respuesta, contador e historial completo."""
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "tool_calls_used": 0,
    }

    if verbose:
        print(f"Consulta: {query}")

    final_state = initial_state
    recursion_limit = 2 * MAX_TOOL_CALLS + 4

    for state in graph.stream(
        initial_state,
        config={"recursion_limit": recursion_limit},
        stream_mode="values",
    ):
        final_state = state
        if not verbose:
            continue

        last_message = state["messages"][-1]
        tool_calls = getattr(last_message, "tool_calls", None) or []

        if tool_calls:
            for call in tool_calls:
                print(f"Herramienta solicitada: {call['name']}")
                print(f"Argumentos: {call['args']}")
        elif isinstance(last_message, ToolMessage):
            print(f"Resultado de {last_message.name}:")
            print(last_message.content)

    final_message = final_state["messages"][-1]
    answer = final_message.content
    used = final_state.get("tool_calls_used", 0)

    if verbose:
        print(f"Respuesta final:\n{answer}")
        print(f"Llamadas ejecutadas: {used}/{MAX_TOOL_CALLS}")

    return {
        "answer": answer,
        "tool_calls_used": used,
        "messages": final_state["messages"],
    }

### Prueba de extremo a extremo

Esta consulta debe seleccionar `resumen_mensual`, ejecutar la herramienta y construir una respuesta final a partir de su JSON.

In [37]:
resultado_demo = run_agent(
    "¿Cuánto ingresé, cuánto gasté y cuál fue mi balance en marzo de 2026?"
)

Consulta: ¿Cuánto ingresé, cuánto gasté y cuál fue mi balance en marzo de 2026?
Herramienta solicitada: resumen_mensual
Argumentos: {'mes': '2026-03'}
Resultado de resumen_mensual:
{
  "ok": true,
  "mes": "2026-03",
  "moneda": "COP",
  "total_ingresos": 10846570,
  "total_egresos_registrados": 7433450,
  "gastos_de_consumo": 6933450,
  "ahorro_e_inversion": 500000,
  "balance_de_caja": 3413120,
  "ahorro_economico_estimado": 3913120,
  "porcentaje_egresos_sobre_ingresos": 68.53,
  "tasa_de_ahorro_estimada": 36.08,
  "cantidad_ingresos": 9,
  "cantidad_egresos": 69
}
Respuesta final:
En marzo de 2026:

- **Ingresos:** 10.846.570 COP
- **Gastos de consumo:** 6.933.450 COP
- **Egresos totales registrados** (incluye 500.000 COP de ahorro e inversión): 7.433.450 COP
- **Balance de caja:** 3.413.120 COP
Llamadas ejecutadas: 1/3


## 8. Casos de prueba

Pendiente: documentar cinco casos con pregunta, resultado esperado, herramienta utilizada, respuesta obtenida y valoración.

### Preguntas habituales que el sistema debería resolver


In [ ]:
resultado_demo1 = run_agent(
    "¿Qué diferencia hay entre gasto de consumo y ahorro/inversión?"
)

Consulta: ¿Qué diferencia hay entre gasto de consumo y ahorro/inversión?
Respuesta final:
El **gasto de consumo** es el dinero destinado a bienes y servicios para satisfacer necesidades o deseos inmediatos (por ejemplo, alimentación, vivienda, transporte, ocio). Una vez realizado, el dinero se “gasta” y no genera un retorno financiero directo.

El **ahorro o la inversión** es el dinero que se aparta del consumo actual para destinarlo a un uso futuro o para hacer crecer el patrimonio (por ejemplo, cuentas de ahorro, depósitos, acciones, fondos de inversión). No se consume de inmediato; su objetivo es generar liquidez futura, seguridad o rendimiento.

En el balance de caja mensual, los egresos totales se dividen en consumo y en aportes a ahorro/inversión; la diferencia entre ingresos y egresos determina si hay superávit o déficit.

Esto no constituye asesoría financiera profesional. Si quieres ver cómo se refleja esto en tus datos, indícame un mes en formato YYYY-MM y puedo consultar el 

In [ ]:
resultado_demo2 = run_agent(
    "¿Puedes analizar datos en tiempo real?"
)

Consulta: ¿Puedes analizar datos en tiempo real?
Respuesta final:
No, no tengo acceso a datos bancarios en tiempo real ni a credenciales de cuentas.

Puedo analizar únicamente la información disponible a través de las herramientas, mes por mes, en formato **YYYY-MM** (por ejemplo, 2026-03). Para hacerlo, indícame el mes que deseas revisar y con qué herramienta: resumen mensual, análisis de gastos o detección de alertas.
Llamadas ejecutadas: 0/3


In [ ]:
resultado_demo3 = run_agent(
    "¿Qué porcentaje de mis ingresos debería ahorrar?"
)

Consulta: ¿Qué porcentaje de mis ingresos debería ahorrar?
Respuesta final:
Para analizar tu tasa de ahorro real y decirte qué porcentaje de tus ingresos estás ahorrando, necesito que me indiques el mes en formato **YYYY-MM** (por ejemplo, 2026-03).

Con esa información puedo:

* Calcular tus ingresos, egresos y tasa de ahorro del mes.
* Detectar si tu ahorro está por debajo del 10 %.
* Mostrarte en qué categorías estás gastando.

Así podrás comparar tu situación actual con referencias generales (comúnmente se sugiere ahorrar entre el 10 % y el 20 % de los ingresos, aunque depende de tus objetivos). Indícame el mes para comenzar.
Llamadas ejecutadas: 0/3


### Pregunta usando herramientas

In [41]:
run_agent(
    "Okey, revisa entonces porfavor abril del 2026"
)

Consulta: Okey, revisa entonces porfavor abril del 2026
Herramienta solicitada: resumen_mensual
Argumentos: {'mes': '2026-04'}
Herramienta solicitada: analizar_gastos
Argumentos: {'mes': '2026-04'}
Herramienta solicitada: detectar_alertas
Argumentos: {'mes': '2026-04'}
Resultado de detectar_alertas:
{
  "ok": true,
  "mes": "2026-04",
  "reglas": {
    "tasa_minima_ahorro_porcentaje": 10,
    "categoria_atipica": "promedio histórico + 2 desviaciones estándar"
  },
  "cantidad_alertas": 3,
  "alertas": [
    {
      "nivel": "alta",
      "tipo": "deficit_de_caja",
      "detalle": "Los egresos superan los ingresos por 595,850 COP."
    },
    {
      "nivel": "media",
      "tipo": "ahorro_bajo",
      "detalle": "La tasa de ahorro estimada es 6.41 %, inferior al umbral de 10 %."
    },
    {
      "nivel": "media",
      "tipo": "categoria_atipica",
      "categoria": "Salud y cuidado personal",
      "monto": 510600,
      "detalle": "El gasto de la categoría supera el promedio histó

{'answer': '**Resumen abril 2026 (COP)**\n\n**Indicadores principales**\n- Ingresos: 7.870.000\n- Egresos registrados: 8.465.850\n- Balance de caja: -595.850 (déficit)\n- Gastos de consumo: 7.365.850\n- Ahorro e inversión: 1.100.000\n- Tasa de ahorro estimada: 6,41 %\n- Egresos sobre ingresos: 107,57 %\n\n**Categorías más relevantes**\n1. Gastos básicos: 4.463.000 (52,72 %)\n2. Ocio: 1.137.500 (13,44 %)\n3. Ahorro e inversión: 1.100.000 (12,99 %)\n4. Préstamos a terceros: 800.000 (9,45 %)\n\n**Alertas detectadas**\n1. **Déficit de caja (alta):** los egresos superan los ingresos en 595.850 COP.\n2. **Ahorro bajo (media):** la tasa de ahorro es 6,41 %, inferior al umbral del 10 %.\n3. **Categoría atípica (media):** Salud y cuidado personal con 510.600 COP, por encima del promedio histórico más dos desviaciones estándar.\n\n*Esta información no constituye asesoría financiera profesional.*',
 'tool_calls_used': 3,
 'messages': [HumanMessage(content='Okey, revisa entonces porfavor abril del

### Pregunta límite

In [46]:
resultado_demo4 = run_agent(
    "Hazme un gráfico que contraste mis ingresos y egresos"
)

Consulta: Hazme un gráfico que contraste mis ingresos y egresos
Respuesta final:
Para poder mostrarte el contraste entre ingresos y egresos, necesito que me indiques el mes que deseas consultar en formato **YYYY-MM** (por ejemplo, 2026-03).

Una vez me lo proporciones, podré obtener los datos del resumen mensual y presentarte la comparación. Ten en cuenta que no puedo generar gráficos directamente, pero sí puedo darte los valores clave y el análisis de esos indicadores.
Llamadas ejecutadas: 0/3


In [48]:
resultado_demo5 = run_agent(
    "¿Cuánto vale un viaje a japón en enero del 2026?"
)

Consulta: ¿Cuánto vale un viaje a japón en enero del 2026?
Respuesta final:
No puedo responder a esa pregunta. Mis herramientas están diseñadas para analizar tus finanzas personales (ingresos, egresos, ahorro, categorías de gasto y alertas mensuales), no para consultar precios de viajes, vuelos o paquetes turísticos.

Si quieres, puedo ayudarte a revisar tu presupuesto mensual o analizar cuánto has destinado a viajes en meses anteriores. Para eso necesitaría que me indiques el mes en formato **YYYY-MM** (por ejemplo, 2026-01).
Llamadas ejecutadas: 0/3


## 9. Resultados, limitaciones y mejoras

**Resultados:** En general,
 - El agente lográ responder correctamente las preguntas asociadas a las herramientas.
 - Tiene un tiempo de cómputo muy razonable.
 - Es honesto cuando no puede responder una pregunta, por lo que no inventa y propone como si puede ayudarte.

**Limitaciones:**
 - No recibe intervalos de meses, solo un mes específico.
 - No puede generar gráficos.
 - No es un agente para ahorro, sirve solo para monitorear y resumir mes a mes. 
 - No puede manipular los datos, hacer transacciones, ni usar datos en tiempo real.
 - Es un agente personal, por lo que todas las respuestas van a estar condicionadas a la base de datos específica.
 
**Mejoras:**
 - Adicionar más herramientas para que el agente pueda hacer tareas más complejas.
 - Que se vuelva un asistente en tiempo real, que no solo pueda dar información de lo que ya pasó, sino también monitorear en tiempo real las finanzas personales.
 - Que tenga información financiera verídica y pueda darle tips financieros al usuario según sus gastos e ingresos específicos.
